In [7]:
import pandas as pd

workbook_url = "https://raw.githubusercontent.com/jozefkip/firstrepo/main/Data%20case%20gas_cross_border%20-%202026.xlsx"
raw_forwards = pd.read_excel(workbook_url, sheet_name="Forwards", header=None)
raw_spot = pd.read_excel(workbook_url, sheet_name="Spot", header=None)

forward_header_row = next(
    index
    for index, row in raw_forwards.iterrows()
    if row.astype(str).str.strip().str.casefold().eq("date").any()
)
forward_date_col = next(
    column
    for column, value in raw_forwards.iloc[forward_header_row].items()
    if str(value).strip().casefold() == "date"
)
forward_group_row = raw_forwards.iloc[forward_header_row - 1]
forward_header = raw_forwards.iloc[forward_header_row]
forward_groups = []
for column, value in forward_group_row.items():
    if pd.isna(value):
        continue
    label = str(value).strip().casefold()
    if "market 2" in label and "market 1" in label:
        group_name = "Spread (€/MWh)"
    elif "market 1" in label:
        group_name = "Market 1 (€/MWh)"
    elif "market 2" in label:
        group_name = "Market 2 (€/MWh)"
    else:
        continue
    forward_groups.append((column, group_name))

forward_parts = []
for group_index, (start_col, group_name) in enumerate(forward_groups):
    end_col = forward_groups[group_index + 1][0] if group_index + 1 < len(forward_groups) else raw_forwards.shape[1]
    contract_cols = [
        column
        for column in range(start_col, end_col)
        if pd.notna(forward_header.iloc[column])
    ]
    part = raw_forwards.iloc[forward_header_row + 1:, [forward_date_col, *contract_cols]].copy()
    part.columns = ["Date", *[str(forward_header.iloc[column]).strip() for column in contract_cols]]
    part = part.melt(id_vars="Date", var_name="Contract", value_name=group_name)
    forward_parts.append(part)

forwards_df = forward_parts[0]
for part in forward_parts[1:]:
    forwards_df = forwards_df.merge(part, on=["Date", "Contract"], how="outer")
forwards_df["Date"] = pd.to_datetime(forwards_df["Date"], errors="coerce")
forwards_df = forwards_df.dropna(subset=["Date"]).sort_values(["Date", "Contract"]).reset_index(drop=True)

spot_header_row = next(
    index
    for index, row in raw_spot.iterrows()
    if row.astype(str).str.strip().str.casefold().eq("date").any()
)
spot_headers = raw_spot.iloc[spot_header_row]
spot_columns = {}
for column, value in spot_headers.items():
    if pd.isna(value):
        continue
    label = str(value).strip().casefold()
    if label == "date":
        spot_columns["Date"] = column
    elif "market 2" in label and "market 1" in label:
        spot_columns["Spread (€/MWh)"] = column
    elif "market 1" in label:
        spot_columns["Market 1 (€/MWh)"] = column
    elif "market 2" in label:
        spot_columns["Market 2 (€/MWh)"] = column

spot_df = raw_spot.iloc[spot_header_row + 1:, list(spot_columns.values())].copy()
spot_df.columns = list(spot_columns)
spot_df["Date"] = pd.to_datetime(spot_df["Date"], errors="coerce")
spot_df = spot_df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

for column in ["Market 1 (€/MWh)", "Market 2 (€/MWh)", "Spread (€/MWh)"]:
    forwards_df[column] = pd.to_numeric(forwards_df[column], errors="coerce")
    spot_df[column] = pd.to_numeric(spot_df[column], errors="coerce")

dataframes = {"Forwards": forwards_df, "Spot": spot_df}
for name, df in dataframes.items():
    print(f"{name}: {df.shape[0]} rows x {df.shape[1]} columns")
    display(df.head())

Forwards: 1724 rows x 5 columns


,Date,Contract,Market 1 (€/MWh),Market 2 (€/MWh),Spread (€/MWh)
0,2024-01-02,Q1-2026,35.093,34.075,-1.018
1,2024-01-02,Q2-2026,29.799,30.838,1.039
2,2024-01-02,Q3-2026,29.085,30.049,0.964
3,2024-01-02,Q4-2026,30.167,30.465,0.298
4,2024-01-03,Q1-2026,36.474,35.128,-1.346


Spot: 618 rows x 4 columns


,Date,Market 1 (€/MWh),Market 2 (€/MWh),Spread (€/MWh)
0,2024-01-01,29.825,29.406653,-0.418347
1,2024-01-02,29.900,29.005747,-0.894253
2,2024-01-03,29.125,28.401440,-0.723560
3,2024-01-04,33.325,31.121312,-2.203688
4,2024-01-05,34.100,32.084273,-2.015727
